# AIDE-TIHM Notebook 1: Explainable Alzheimer's Clinical Risk Assessment

This notebook reproduces the clinical risk classification task using validated psychometric batteries and biomarkers (OASIS/ADNI reference schema).


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add src to path
sys.path.append(os.path.abspath("../src"))

from data_preprocessing import load_or_create_datasets, patient_level_split
from feature_engineering import extract_clinical_features
from alzheimer_model import train_alzheimer_models, evaluate_alzheimer_test
from explainability import SHAPExplainerEngine


## 1. Load Clinical Cohort Data


In [ ]:
clinical_df, _ = load_or_create_datasets(data_dir="../data")
print("Clinical Dataset Shape:", clinical_df.shape)
print("Unique Participants:", clinical_df['patient_id'].nunique())
clinical_df.head()


## 2. Zero-Leakage Participant-Level Splitting


In [ ]:
splits = patient_level_split(clinical_df, patient_col='patient_id', target_col='diagnosis', test_size=0.20, val_size=0.15)
train_df = splits['train_df']
val_df = splits['val_df']
test_df = splits['test_df']

print("Train Patients:", train_df['patient_id'].nunique(), "| Samples:", len(train_df))
print("Val Patients:  ", val_df['patient_id'].nunique(), "| Samples:", len(val_df))
print("Test Patients: ", test_df['patient_id'].nunique(), "| Samples:", len(test_df))


## 3. Feature Extraction & Engineering


In [ ]:
X_train, y_train, feature_cols, scaler = extract_clinical_features(train_df, is_training=True)
X_val, y_val, _, _ = extract_clinical_features(val_df, is_training=False, scaler=scaler)
X_test, y_test, _, _ = extract_clinical_features(test_df, is_training=False, scaler=scaler)

print(f"Features Extracted ({len(feature_cols)}):", feature_cols[:10])


## 4. Model Training & Validation Selection


In [ ]:
models, best_model_name, val_scores = train_alzheimer_models(X_train, y_train, X_val, y_val)
print("Validation AUC Scores:", val_scores)
print("Selected Best Model:", best_model_name)


## 5. Held-Out Test Evaluation


In [ ]:
test_eval = evaluate_alzheimer_test(models, best_model_name, X_test, y_test)
best_metrics = test_eval['best_test_metrics']
print(f"Test Evaluation for {best_model_name}:")
for k, v in best_metrics.items():
    if k != 'confusion_matrix':
        print(f"  {k}: {v}")


## 6. Explainability with SHAP


In [ ]:
best_clf = models[best_model_name]
explainer = SHAPExplainerEngine(best_clf, feature_cols, background_data=X_train[:100])

# Global Summary Plot
explainer.plot_summary(X_test, save_path="../results/shap_summary_nb.png")

# Local Explanation for Patient 0
sample_raw = test_df.iloc[0].to_dict()
local_exp = explainer.explain_instance(X_test[0], raw_feature_values=sample_raw, top_k=5)
print("Top 5 Contributing Factors:")
for f in local_exp['top_k_features']:
    print(f"  {f['feature']}: SHAP={f['shap_value']:+.4f} (Value={f['feature_value']}) -> {f['direction']}")
